# 07 — Model Comparison

This notebook compares ARIMA, ETS, Prophet, LSTM, and Ensemble metrics across all stocks.
It saves aggregate tables, best-model selections, and a short rationale for final model choice.

In [ ]:
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd()
REPORT_DIR = ROOT / 'outputs' / 'reports'
PRED_DIR = ROOT / 'outputs' / 'predictions'
CHART_DIR = ROOT / 'outputs' / 'charts' / 'comparison'
CHART_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.preprocessor import SELECTED, NAMES

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

# ── Load all model metrics ─────────────────────────────────────
model_files = {
    'ARIMA': REPORT_DIR / 'arima_metrics.csv',
    'ETS': REPORT_DIR / 'ets_metrics.csv',
    'Prophet': REPORT_DIR / 'prophet_metrics.csv',
    'LSTM': REPORT_DIR / 'lstm_metrics.csv',
}

all_metrics = []
for model_name, filepath in model_files.items():
    if filepath.exists():
        df = pd.read_csv(filepath)
        df['Model_Source'] = model_name
        all_metrics.append(df)
        print(f'✅ Loaded {model_name} metrics ({len(df)} rows)')
    else:
        print(f'⚠️  {model_name} metrics not found: {filepath}')

if not all_metrics:
    print('ERROR: No model metrics files found. Run notebooks 02-05 first.')
else:
    # Combine all metrics
    combined = pd.concat(all_metrics, ignore_index=True)
    
    # Summary statistics by model
    model_summary = combined.groupby('Model_Source')[['RMSE', 'MAE', 'MAPE_%', 'DA_%', 'R2']].agg(['mean', 'std', 'min', 'max']).round(4)
    print('\n' + '='*80)
    print('MODEL PERFORMANCE SUMMARY')
    print('='*80)
    display(model_summary)
    
    # Best model by ticker
    best_by_ticker = combined.loc[combined.groupby('Ticker')['RMSE'].idxmin()]
    print('\n' + '='*80)
    print('BEST MODEL PER STOCK (Lowest RMSE)')
    print('='*80)
    display(best_by_ticker[['Ticker', 'Company', 'Model_Source', 'RMSE', 'MAE', 'MAPE_%', 'DA_%']])
    
    # Overall ranking
    print('\n' + '='*80)
    print('OVERALL MODEL RANKING (Mean RMSE across all stocks)')
    print('='*80)
    model_rank = combined.groupby('Model_Source')['RMSE'].mean().sort_values().reset_index()
    model_rank.columns = ['Model', 'Mean_RMSE']
    model_rank['Rank'] = range(1, len(model_rank) + 1)
    display(model_rank)
    
    # Visualization
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # RMSE by model
    ax = axes[0, 0]
    model_rmse = combined.groupby('Model_Source')['RMSE'].mean().sort_values()
    ax.barh(model_rmse.index, model_rmse.values, color='steelblue')
    ax.set_xlabel('Mean RMSE')
    ax.set_title('Model Comparison: Mean RMSE')
    ax.grid(axis='x', alpha=0.3)
    
    # DA% by model
    ax = axes[0, 1]
    model_da = combined.groupby('Model_Source')['DA_%'].mean().sort_values(ascending=False)
    ax.barh(model_da.index, model_da.values, color='darkorange')
    ax.set_xlabel('Mean Directional Accuracy %')
    ax.set_title('Model Comparison: Directional Accuracy')
    ax.grid(axis='x', alpha=0.3)
    
    # MAPE% by model
    ax = axes[1, 0]
    model_mape = combined.groupby('Model_Source')['MAPE_%'].mean().sort_values()
    ax.barh(model_mape.index, model_mape.values, color='mediumseagreen')
    ax.set_xlabel('Mean MAPE %')
    ax.set_title('Model Comparison: MAPE')
    ax.grid(axis='x', alpha=0.3)
    
    # R² by model
    ax = axes[1, 1]
    model_r2 = combined.groupby('Model_Source')['R2'].mean().sort_values(ascending=False)
    ax.barh(model_r2.index, model_r2.values, color='crimson')
    ax.set_xlabel('Mean R²')
    ax.set_title('Model Comparison: R²')
    ax.grid(axis='x', alpha=0.3)
    
    fig.tight_layout()
    fig.savefig(CHART_DIR / 'model_comparison_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Save summary
    model_summary.to_csv(REPORT_DIR / 'model_comparison_summary.csv')
    best_by_ticker.to_csv(REPORT_DIR / 'best_model_per_ticker.csv', index=False)
    
    print('\n✅ Model comparison complete. Saved:')
    print(f'   - {REPORT_DIR / "model_comparison_summary.csv"}')
    print(f'   - {REPORT_DIR / "best_model_per_ticker.csv"}')
    print(f'   - {CHART_DIR / "model_comparison_summary.png"}')